# Evaluación de políticas de asignación

Implementa la Sección 5 de la tesis con los puntajes fuera de municipio de `evaluacion_modelos.ipynb` (validación cruzada de la política, Sección 5.2: cada municipio se puntúa con un modelo que no vio sus resultados).

- **Políticas** (Sección 5.1): sin información, menor costo, mayor probabilidad de hallazgo, probabilidad de hallazgo por costo y mochila.
- **Presupuesto**: la política se aplica dentro de cada municipio, con una fracción $f$ del presupuesto histórico del municipio (el costo de intervenir todos sus sitios), y los hallazgos se suman entre municipios. Esto refleja que la UBPD planifica por regiones.
- **Costos**: (a) homogéneos, $c_i = 1$; (b) heterogéneos ilustrativos, $c_i = 1 + d_i$, con $d_i$ la distancia a la vía más cercana en km (una unidad por misión más una por kilómetro de acceso). Los costos reales por diligencia están pendientes.
- **Evaluación sobre diligencias observadas** (Sección 5.3): universo = sitios con resultado; se cuentan los hallazgos observados.
- **Simulación territorial** (Sección 5.4): universo = todos los sitios candidatos, incluidos los que no tienen resultado; $G_i \sim \text{Bernoulli}(p_i^{eval})$, con $p_i^{eval}$ de un modelo distinto del que construye la política, recalibrado sobre los sitios observados.
- **Sin información** (Definición 1): con costos homogéneos elige sitios al azar (se reporta su valor esperado exacto); con costos heterogéneos elige los más baratos.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import sys

sys.path.append('../src')
from politicas import POLITICAS, seleccion

PATH_PROCESSED = Path("../data/Processed/")
PATH_FIGURAS = Path("../data/images/")

MODELOS = ['Maxent (fondo)', 'Maxent (prospecciones)', 'XGBoost', 'Random Forest', 'GLM']
MODELO_POLITICA = 'GLM'                         # mayor AUC promedio fuera de municipio (Cuadro 3)
MODELOS_EVALUACION = ['Random Forest', 'XGBoost']  # distintos del modelo de la política
FRACCION_TABLA = 0.3                            # coherente con k = 30% del Cuadro 3
FRACCIONES = np.round(np.arange(0, 1.0001, 0.01), 2)
N_SIMULACIONES = 2000
SEMILLA = 42

pred = pd.read_csv(PATH_PROCESSED / "predicciones_fuera_municipio.csv", dtype={'municipio': str})
pred['costo_homogeneo'] = 1.0
pred['costo_heterogeneo'] = 1.0 + pred['dist_vias'] / 1000
observados = pred[pred['label'].notna()].reset_index(drop=True)

print(f"Sitios candidatos: {len(pred)}  |  con resultado: {len(observados)}")
print(f"Costo heterogéneo: media {pred['costo_heterogeneo'].mean():.2f}, "
      f"mediana {pred['costo_heterogeneo'].median():.2f}, máx. {pred['costo_heterogeneo'].max():.1f}")


In [ ]:
# ============================================================
# EVALUACIÓN SOBRE DILIGENCIAS OBSERVADAS
# ============================================================
def evaluar_observado(modelo, nombre, col_costo, f):
    a = seleccion(observados, modelo, nombre, col_costo, f)
    return {'costo': (a * observados[col_costo]).sum(), 'hallazgos': (a * observados['label']).sum()}

def curva(evaluar, modelo, nombre, col_costo):
    return pd.DataFrame([{'f': f, **evaluar(modelo, nombre, col_costo, f)} for f in FRACCIONES])

def equivalente_presupuestario(curva_m, curva_0, presupuesto_total, p_media=None):
    """Delta B^eq(f): presupuesto adicional que necesita la política sin información para
    igualar los hallazgos del modelo con fracción f (0 si ya los iguala).

    Con costos homogéneos (p_media dada) se usa la Proposición 3(iii): Delta K = V_pred / p_media.
    Esto corresponde a la definición cuando la autoridad sin información puede repartir las
    intervenciones adicionales entre municipios; repartirlas en proporción al presupuesto de
    cada municipio introduce saltos artificiales en los municipios con uno o dos sitios."""
    if p_media is not None:
        v = curva_m['hallazgos'].values - curva_0['hallazgos'].values
        return np.maximum(v, 0) / p_media
    h0 = curva_0['hallazgos'].values
    delta = []
    for f, hm in zip(curva_m['f'], curva_m['hallazgos']):
        alcanzan = np.where(h0 >= hm - 1e-9)[0]
        f_eq = curva_0['f'].values[alcanzan[0]] if len(alcanzan) else np.nan
        delta.append(max(f_eq - f, 0) * presupuesto_total)
    return np.array(delta)


In [ ]:
# ============================================================
# SIMULACIÓN TERRITORIAL
# ============================================================
# p_eval: puntajes fuera de municipio del modelo de evaluación, recalibrados con una regresión
# logística sobre los sitios observados (Platt). RF y XGBoost se entrenan con pesos de clase
# balanceados, de modo que sus probabilidades no están en la escala de la tasa de hallazgo.
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

p_eval = {}
for e in MODELOS_EVALUACION:
    cal = LogisticRegression().fit(logit(observados[e].values)[:, None], observados['label'])
    p_eval[e] = cal.predict_proba(logit(pred[e].values)[:, None])[:, 1]
    print(f"p_eval {e:<14}: media {p_eval[e].mean():.3f}  |  desv. est. {p_eval[e].std():.3f}")

rng = np.random.default_rng(SEMILLA)
G = {e: rng.random((N_SIMULACIONES, len(pred))) < p_eval[e] for e in MODELOS_EVALUACION}

def evaluar_simulado(modelo, nombre, col_costo, f, intervalos=False):
    a = seleccion(pred, modelo, nombre, col_costo, f)
    esperado = np.mean([(a * p_eval[e]).sum() for e in MODELOS_EVALUACION])
    salida = {'costo': (a * pred[col_costo]).sum(), 'hallazgos': esperado}
    if intervalos:
        # Para la política al azar, se sortea un subconjunto en cada simulación
        draws = []
        for e in MODELOS_EVALUACION:
            if np.any((a > 0) & (a < 1)):
                sel = np.zeros((N_SIMULACIONES, len(pred)), bool)
                for _, idx in pred.groupby('municipio').indices.items():
                    k = int(round(a[idx][0] * len(idx)))
                    orden = np.argsort(rng.random((N_SIMULACIONES, len(idx))), axis=1)[:, :k]
                    sel[np.arange(N_SIMULACIONES)[:, None], idx[orden]] = True
                draws.append((G[e] & sel).sum(axis=1))
            else:
                draws.append(G[e][:, a == 1].sum(axis=1))
        salida['draws'] = np.concatenate(draws)
    return salida


In [ ]:
# ============================================================
# CUADRO DE POLÍTICAS (fracción FRACCION_TABLA del presupuesto histórico)
# ============================================================
filas = []
for evaluacion, evaluar, df in [('Diligencias observadas', evaluar_observado, observados),
                                ('Simulación territorial', evaluar_simulado, pred)]:
    for escenario, col_costo in [('Costos homogéneos', 'costo_homogeneo'),
                                 ('Costos heterogéneos', 'costo_heterogeneo')]:
        presupuesto_total = df[col_costo].sum()
        p_media = ((df['label'].mean() if evaluacion == 'Diligencias observadas'
                    else np.mean([p_eval[e].mean() for e in MODELOS_EVALUACION]))
                   if col_costo == 'costo_homogeneo' else None)
        curva_0 = curva(evaluar, MODELO_POLITICA, 'Sin información', col_costo)
        base = None
        for nombre in POLITICAS:
            if evaluacion == 'Simulación territorial':
                r = evaluar_simulado(MODELO_POLITICA, nombre, col_costo, FRACCION_TABLA, intervalos=True)
            else:
                r = evaluar(MODELO_POLITICA, nombre, col_costo, FRACCION_TABLA)
            if nombre == 'Sin información':
                base = r
            curva_m = curva(evaluar, MODELO_POLITICA, nombre, col_costo)
            i = int(np.where(np.isclose(curva_m['f'], FRACCION_TABLA))[0][0])
            fila = {'evaluacion': evaluacion, 'escenario': escenario, 'politica': nombre,
                    'costo': r['costo'], 'hallazgos': r['hallazgos'],
                    'costo_por_hallazgo': r['costo'] / r['hallazgos'] if r['hallazgos'] > 0 else np.nan,
                    'delta_B_eq': equivalente_presupuestario(curva_m, curva_0, presupuesto_total, p_media)[i]}
            if 'draws' in r:
                fila['hallazgos_p5'], fila['hallazgos_p95'] = np.percentile(r['draws'], [5, 95])
                # Probabilidad de superar a la política sin información (empates cuentan la mitad)
                fila['prob_supera_sin_info'] = (np.nan if nombre == 'Sin información' else
                                                np.mean(r['draws'] > base['draws'])
                                                + 0.5 * np.mean(r['draws'] == base['draws']))
            filas.append(fila)

df_politicas = pd.DataFrame(filas)
df_politicas.to_csv(PATH_PROCESSED / "politicas.csv", index=False)
pd.set_option('display.width', 250)
print(f"Modelo de la política: {MODELO_POLITICA}  |  presupuesto: {FRACCION_TABLA:.0%} del histórico por municipio")
print(df_politicas.round(2).to_string(index=False))


In [ ]:
# ============================================================
# VALOR DE LA PREDICCIÓN Y EQUIVALENTE PRESUPUESTARIO
# (diligencias observadas, costos homogéneos)
# ============================================================
col_costo = 'costo_homogeneo'
presupuesto_total = observados[col_costo].sum()
curva_0 = curva(evaluar_observado, MODELO_POLITICA, 'Sin información', col_costo)
curvas = {}
for m in MODELOS:
    cm = curva(evaluar_observado, m, 'Mochila', col_costo)
    cm['V_pred'] = cm['hallazgos'] - curva_0['hallazgos']
    cm['delta_B_eq'] = equivalente_presupuestario(cm, curva_0, presupuesto_total, observados['label'].mean())
    curvas[m] = cm

resumen = pd.DataFrame([{
    'modelo': m,
    'V_pred(0)': c['V_pred'].iloc[0],
    'V_pred(1)': c['V_pred'].iloc[-1],
    'max V_pred': c['V_pred'].max(),
    'f en el máximo': c.loc[c['V_pred'].idxmax(), 'f'],
    'min V_pred': c['V_pred'].min(),
    'V_pred(0.3)': c.loc[np.isclose(c['f'], 0.3), 'V_pred'].iloc[0],
    'max Delta B_eq': c['delta_B_eq'].max(),
} for m, c in curvas.items()])
print(f"Hallazgos con todo el presupuesto: {curva_0['hallazgos'].iloc[-1]:.0f} de {len(observados)} sitios")
print(resumen.round(2).to_string(index=False))

pd.concat([c.assign(modelo=m) for m, c in curvas.items()]).to_csv(
    PATH_PROCESSED / "curvas_valor_prediccion.csv", index=False)


In [ ]:
# ============================================================
# REFERENCIA: PUNTAJES ALEATORIOS
# ============================================================
# Banda de V_pred(f) que se obtiene con puntajes al azar, para leer si el valor de cada
# modelo se distingue del que produciría un ordenamiento sin información.
N_ALEATORIOS = 500
rng_ref = np.random.default_rng(SEMILLA)
ref = observados[['municipio', 'label', 'costo_homogeneo']].copy()
v_aleatorio = []
for _ in range(N_ALEATORIOS):
    ref['aleatorio'] = rng_ref.random(len(ref))
    h = np.array([(seleccion(ref, 'aleatorio', 'Mochila', 'costo_homogeneo', f) * ref['label']).sum()
                  for f in FRACCIONES])
    v_aleatorio.append(h - curva_0['hallazgos'].values)
v_aleatorio = np.array(v_aleatorio)
banda = np.percentile(v_aleatorio, [5, 95], axis=0)
print(f"Puntajes aleatorios: percentil 95 del máximo de V_pred = "
      f"{np.percentile(v_aleatorio.max(axis=1), 95):.2f}  |  "
      f"banda 5-95 en f = 0.3: [{banda[0][30]:.2f}, {banda[1][30]:.2f}]")
for m, c in curvas.items():
    fuera = np.mean((c['V_pred'].values > banda[1]) | (c['V_pred'].values < banda[0]))
    p_max = np.mean(v_aleatorio.max(axis=1) >= c['V_pred'].max())
    print(f"  {m:<24} fracción de presupuestos fuera de la banda: {fuera:.2f}  |  "
          f"P(máximo aleatorio >= máximo del modelo): {p_max:.2f}")

# ============================================================
# FIGURA
# ============================================================
COLORES = {'Maxent (fondo)': '#2a78d6', 'Maxent (prospecciones)': '#eb6834',
           'XGBoost': '#1baf7a', 'Random Forest': '#eda100', 'GLM': '#e87ba4'}
ESTILOS = {'Maxent (fondo)': '-', 'Maxent (prospecciones)': '--',
           'XGBoost': '-.', 'Random Forest': ':', 'GLM': '-'}
ETIQUETAS = {'Maxent (fondo)': 'Maxent (fondo del territorio)',
             'Maxent (prospecciones)': 'Maxent (prospecciones negativas)',
             'XGBoost': 'XGBoost', 'Random Forest': 'Random Forest', 'GLM': 'GLM'}
TINTA, TINTA_SEC, REJILLA = '#1f1f1e', '#5f5e5a', '#e4e3dd'

plt.rcParams.update({'font.family': 'serif', 'font.size': 9, 'axes.edgecolor': TINTA_SEC,
                     'axes.labelcolor': TINTA, 'xtick.color': TINTA_SEC, 'ytick.color': TINTA_SEC})
fig, ejes = plt.subplots(1, 3, figsize=(10, 3.8))
x = FRACCIONES * 100

ax = ejes[0]
ax.plot(x, curvas[MODELO_POLITICA]['hallazgos'], color=COLORES[MODELO_POLITICA], lw=2,
        label=f'Con modelo ({MODELO_POLITICA}), $H_M$')
ax.plot(x, curva_0['hallazgos'], color=TINTA_SEC, lw=2, ls='--', label='Sin información, $H_0$')
ax.set_title('(a) Hallazgos según el presupuesto', loc='left', fontsize=9, color=TINTA)
ax.set_ylabel('Hallazgos')
ax.legend(frameon=False, fontsize=8, loc='upper left')

for ax, col, titulo, ylab in [(ejes[1], 'V_pred', '(b) Valor de la predicción', '$V_{pred}$ (hallazgos)'),
                              (ejes[2], 'delta_B_eq', '(c) Equivalente presupuestario',
                               '$\\Delta B^{eq}$ (intervenciones)')]:
    if col == 'V_pred':
        ax.fill_between(x, banda[0], banda[1], color=REJILLA, lw=0,
                        label='Puntajes aleatorios (5-95%)')
    for m in MODELOS:
        ax.plot(x, curvas[m][col], color=COLORES[m], ls=ESTILOS[m],
                lw=1.6 if col == 'V_pred' else 1.1, label=ETIQUETAS[m])
    ax.axhline(0, color=TINTA_SEC, lw=0.8)
    ax.set_title(titulo, loc='left', fontsize=9, color=TINTA)
    ax.set_ylabel(ylab)
manijas, etiquetas = ejes[1].get_legend_handles_labels()
fig.legend(manijas, etiquetas, frameon=False, fontsize=8, ncol=3, loc='lower center',
           bbox_to_anchor=(0.5, -0.02))

for ax in ejes:
    ax.set_xlabel('Presupuesto (% del histórico)')
    ax.grid(True, color=REJILLA, lw=0.6)
    ax.set_axisbelow(True)
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)

fig.tight_layout(rect=(0, 0.12, 1, 1))
PATH_FIGURAS.mkdir(exist_ok=True)
fig.savefig(PATH_FIGURAS / "valor_prediccion.pdf", bbox_inches='tight')
fig.savefig(PATH_FIGURAS / "valor_prediccion.png", dpi=300, bbox_inches='tight')
print("✓ Figura guardada")
